# Ultralytics detectors: YOLO11 (one-stage, anchor-free) and RT-DETR (query-based)

COCO-pretrained models fine-tuned on every 5th training frame (every 2nd in the `stride2` run). In training images, ignore regions are greyed out and boxes more than half inside them are dropped. Validation and test frames are predicted as they are; the evaluator skips detections in ignore regions (`iscrowd`). Each run is one entry in `EXPERIMENTS`, and `RUN` picks it.

Three options target small objects and the small validation split:

- `tile`: training also uses crops of `tile` px cut from the original-resolution frames, so small objects keep all their pixels (a 1920 px frame shrunk to 1280 loses a third of them). Prediction runs on the whole frame and on the same crops, merged by NMS.
- `temporal`: the three input channels are the grey frames t-k, t and t+k of the clip instead of RGB. Moving objects leave coloured edges that a single frame does not have; the input stays 3-channel, so the COCO weights still load. Hue and saturation augmentation is off, since it would mix the frames.
- `RUN = "<experiment>_fold<k>"`: the 56 train and 7 validation clips are pooled and split by clip into 5 folds; the run trains on 4 folds and is scored on the held-out one, so the validation clips are also used for training.

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from ultralytics import RTDETR, YOLO

ROOT = Path.cwd() if Path("src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.data import load, make_yolo_dataset, select_frames, split_clips, write_frames
from src.evaluate import evaluate, to_coco
from src.predict import predict_tiled, predict_yolo
from src.submission import write_submission

KAGGLE = Path("/kaggle/input").exists()
DATA = next(Path("/kaggle/input").rglob("train/annotations_coco.json")).parents[1] if KAGGLE else ROOT / "data/raw"
WORK = Path("/kaggle/working") if KAGGLE else ROOT / "runs/local"
PREPARED = Path("/kaggle/temp/yolo") if KAGGLE else WORK / "yolo"
FRAMES = Path("/kaggle/temp/frames") if KAGGLE else WORK / "frames"  # temporal inputs for validation and test
GPUS = torch.cuda.device_count()
DEVICE = list(range(GPUS)) if GPUS > 1 else 0 if GPUS else "mps"  # a list of GPUs means DDP

EXPERIMENTS = {  # batch is per GPU; 1280 px with batch 8 takes 13.6 of the T4's 15 GB
    "yolo11s_1280_20ep": dict(model="yolo11s.pt", imgsz=1280, epochs=20, batch=8, optimizer="SGD", lr0=0.01),
    "yolo11s_1280_5ep_lr001": dict(model="yolo11s.pt", imgsz=1280, epochs=5, batch=8, optimizer="SGD", lr0=0.001),
    "yolo11s_1536_5ep_lr001": dict(model="yolo11s.pt", imgsz=1536, epochs=5, batch=4, optimizer="SGD", lr0=0.001),
    "yolo11s_1280_10ep_lr001": dict(model="yolo11s.pt", imgsz=1280, epochs=10, batch=8, optimizer="SGD", lr0=0.001),
    "rtdetr-l_960_10ep": dict(model="rtdetr-l.pt", imgsz=960, epochs=10, batch=4, optimizer="AdamW", lr0=0.0001),
    "rtdetr-l_960_10ep_lr0002": dict(model="rtdetr-l.pt", imgsz=960, epochs=10, batch=4, optimizer="AdamW", lr0=0.0002),
    "yolo11s_1280_5ep_lr001_stride2": dict(model="yolo11s.pt", imgsz=1280, epochs=5, batch=8, optimizer="SGD", lr0=0.001,
                                           train_stride=2),
    "rtdetr-l_960_20ep_lr0002": dict(model="rtdetr-l.pt", imgsz=960, epochs=20, batch=4, optimizer="AdamW", lr0=0.0002),
    "yolo11s_1280_3ep_lr001_tile960": dict(model="yolo11s.pt", imgsz=1280, epochs=3, batch=8, optimizer="SGD", lr0=0.001, tile=960),
    "yolo11s_1280_5ep_lr001_temporal2": dict(model="yolo11s.pt", imgsz=1280, epochs=5, batch=8, optimizer="SGD", lr0=0.001, temporal=2),
    "yolo11m_1280_3ep_lr001_tile960": dict(model="yolo11m.pt", imgsz=1280, epochs=3, batch=4, optimizer="SGD", lr0=0.001, tile=960),
}
RUN = "yolo11s_1280_5ep_lr001"
BASE, _, FOLD = RUN.partition("_fold")
CFG = {"train_stride": 5, "curve_stride": 10, "seed": 42, "tile": None, "temporal": 0, **EXPERIMENTS[BASE],
       "fold": int(FOLD) if FOLD else None}
BATCH = CFG["batch"] * max(GPUS, 1)
Model = RTDETR if CFG["model"].startswith("rtdetr") else YOLO
RUN_DIR = next(Path("/kaggle/input").rglob(RUN), None) if KAGGLE else None  # finished run attached as input
print(RUN, "|", GPUS, "x", torch.cuda.get_device_name(0) if GPUS else DEVICE)

## Data

Training frames are resized once so the long side equals `imgsz`, which makes JPEG decoding in the data loader cheap. File names become relative to the data root, so pooled train and validation clips share one root. Temporal runs write their validation and test inputs once, at original resolution.

In [ ]:
train_coco, val_coco = load(DATA / "train/annotations_coco.json"), load(DATA / "val/annotations_coco.json")
test = pd.read_csv(DATA / "test_images.csv")
test_images = [dict(id=r.image_id, file_name=r.file_name, video=r.clip, frame_id=r.frame) for r in test.itertuples()]
train_coco, val_coco = split_clips(train_coco, val_coco, CFG["fold"])
t = time.time()
data_yaml = make_yolo_dataset(train_coco, val_coco, DATA, PREPARED, CFG["train_stride"], CFG["curve_stride"], CFG["imgsz"],
                              CFG["tile"], CFG["temporal"])
if CFG["temporal"]:
    write_frames(val_coco["images"] + test_images, DATA, FRAMES, CFG["temporal"])
SOURCE = FRAMES if CFG["temporal"] else DATA
train_frames = len(select_frames(train_coco, CFG["train_stride"]))
del train_coco  # 1.1M train boxes are not needed after export
print(f"{train_frames} train frames, {len(list((PREPARED / 'images/train').glob('*.jpg')))} training images, prepared in {time.time() - t:.0f}s")

## Training

SGD for YOLO, AdamW for RT-DETR, cosine decay, AMP, mosaic switched off for the last quarter of the epochs. Ultralytics validates every epoch on every 10th validation frame (masked); that draws the training curve and picks `best.pt`. Reported metrics come from the full validation split below. If a finished run's output is attached as input, training is skipped.

In [ ]:
if RUN_DIR is None:
    Model(CFG["model"]).train(data=str(data_yaml), imgsz=CFG["imgsz"], epochs=CFG["epochs"], batch=BATCH, device=DEVICE,
                              seed=CFG["seed"], optimizer=CFG["optimizer"], lr0=CFG["lr0"], cos_lr=True,
                              close_mosaic=max(CFG["epochs"] // 4, 1),
                              cache="ram" if GPUS == 1 else False,  # each DDP process would hold its own copy
                              **(dict(hsv_h=0, hsv_s=0) if CFG["temporal"] else {}),
                              project=str(WORK), name=RUN, exist_ok=True, plots=False)
    RUN_DIR = WORK / RUN
print(RUN_DIR)

In [ ]:
curve = pd.read_csv(RUN_DIR / "results.csv")
train_hours = curve.time.iloc[-1] / 3600
plt.figure(figsize=(6, 3.5))
plt.plot(curve.epoch, curve["metrics/mAP50-95(B)"], marker="o", label="mAP@[.5:.95]")
plt.plot(curve.epoch, curve["metrics/mAP50(B)"], marker="o", label="AP50")
plt.xlabel("epoch")
plt.title(f"{RUN}: Ultralytics val, every {CFG['curve_stride']}th frame")
plt.legend()
plt.show()
print(f"training time: {train_hours:.2f} h")

## Validation: all validation frames (the held-out fold for fold runs), best checkpoint

In [ ]:
model = Model(RUN_DIR / "weights/best.pt")
predict = (lambda m, images, imgsz: predict_tiled(m, images, imgsz, CFG["tile"])) if CFG["tile"] else predict_yolo
val_frames = [(i["id"], SOURCE / i["file_name"]) for i in select_frames(val_coco)]
if CFG["tile"]:  # the same weights without crops, to separate tiled training from tiled prediction
    print("whole frames only:", {k: round(v, 4) for k, v in evaluate(val_coco, predict_yolo(model, val_frames, CFG["imgsz"])[0]).items()})
val_dets, ms_img = predict(model, val_frames, CFG["imgsz"])
metrics = evaluate(val_coco, val_dets)
(WORK / "val_predictions.json").write_text(json.dumps(to_coco(val_dets)))
pd.DataFrame([{**metrics, "ms_img": ms_img, "empty_images": sum(len(b) == 0 for b, _, _ in val_dets.values())}]).round(4)

## Test predictions

In [ ]:
test_dets, _ = predict(model, [(r.image_id, SOURCE / r.file_name) for r in test.itertuples()], CFG["imgsz"])
write_submission(test, test_dets, WORK / "submission.csv")
row = dict(run_id=RUN, status="completed", model=CFG["model"], seed=CFG["seed"], train_stride=CFG["train_stride"],
           train_frames=train_frames, val_frames=len(val_frames), epochs=CFG["epochs"], imgsz=CFG["imgsz"], **metrics,
           ms_img=ms_img, gpu_hours=train_hours, public_lb=None,
           notes=f'{max(GPUS, 1)} GPU, batch {BATCH}, {CFG["optimizer"]} {CFG["lr0"]} cosine, best.pt'
                 + (f', tile {CFG["tile"]}' if CFG['tile'] else '') + (f', temporal {CFG["temporal"]}' if CFG['temporal'] else '')
                 + (f', fold {CFG["fold"]} of 5: scored on its held-out clips' if CFG['fold'] is not None else ''))
pd.DataFrame([row]).to_csv(WORK / "experiment.csv", index=False)
print("DONE:", len(test), "rows in submission.csv")